In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# import all dependencies

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import cv
from sklearn.metrics import accuracy_score
from sklearn import preprocessing

# use inline matplotlib for jupyter session

%matplotlib inline

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Datasets**

We use 5 datasets: Eye Movement, Forest Cover, Churn Modelling, Rossman Store Sales, and Higgs Boson datasets. All common tabular datasets that are popular and often explored on Kaggle.

# **Eye Movements**

In [ ]:
#reading Inputs
Eye_Movements_df=pd.read_csv('/kaggle/input/eye-movements/eye_movements.csv')
#items_df
Eye_Movements_df.describe().T

In [ ]:
Eye_Movements_train_df=pd.read_csv('/kaggle/input/eye-movements/eye_movements.csv')
Eye_Movements_train_df

# **Forest Cover Type**

In [ ]:
#reading Inputs
Forest_Cover_Type_df=pd.read_csv('/kaggle/input/forest-cover-type-prediction/sampleSubmission.csv')
#items_df
Forest_Cover_Type_df.describe().T

In [ ]:
Cover_Type_test_df=pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')
Cover_Type_test3_df=pd.read_csv('/kaggle/input/forest-cover-type-prediction/test3.csv')
Cover_Type_train_df=pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
Cover_Type_train_df

# **Churn Modelling**

In [ ]:
#reading Inputs
Churn_Modelling_df=pd.read_csv('/kaggle/input/churn-modelling/Churn_Modelling.csv')
#items_df
Churn_Modelling_df.describe().T

In [ ]:
Churn_Modelling_train_df=pd.read_csv('/kaggle/input/churn-modelling/Churn_Modelling.csv')
Churn_Modelling_train_df

# **Rossmann**

In [ ]:
#reading Inputs
Rossmann_items_df=pd.read_csv('/kaggle/input/rossmann/items.csv')
#items_df
Rossmann_items_df.describe().T

In [ ]:
Rossmann_item_categories_df=pd.read_csv('/kaggle/input/rossmann/item_categories.csv')
Rossmann_shops_df=pd.read_csv('/kaggle/input/rossmann/shops.csv')
Rossmann_sales_train_df=pd.read_csv('/kaggle/input/rossmann/sales_train.csv')
Rossmann_sales_train_df

# **Higgs Boson**

In [ ]:
#reading Inputs
Higgs_Boson_df=pd.read_csv('/kaggle/input/higgs-boson/Higgs Boson/random_submission.csv')
#items_df
Higgs_Boson_df.describe().T

In [ ]:
Higgs_Boson_test_df=pd.read_csv('/kaggle/input/higgs-boson/Higgs Boson/test.csv')
Higgs_Boson_train_df=pd.read_csv('/kaggle/input/higgs-boson/Higgs Boson/training.csv')
Higgs_Boson_train_df

# **Exploratory Data Analysis (EDA)**

# **Rossmann**

In [ ]:
#distribution of item ID
plt.figure(figsize=(6,5))
plt.hist(Rossmann_sales_train_df['item_id'])
plt.show

In [ ]:
Rossmann_sales_train_df.groupby(['item_id']).nunique()

In [ ]:
Rossmann_sales_train_df[['shop_id','item_id','item_price','item_cnt_day']].corr()

In [ ]:
#EDA on train data
Rossmann_sales_train_df.groupby(['shop_id']).nunique()

In [ ]:
#Distribution of shop_id
plt.figure(figsize=(6,5))
plt.hist(Rossmann_sales_train_df['shop_id'])
plt.show

In [ ]:
monthly = Rossmann_sales_train_df.groupby(['date_block_num'])['item_cnt_day'].sum().reset_index()

In [ ]:
fig = plt.figure(figsize = (20, 10))
ax = fig.add_axes([0, 0, 1, 1])
x = monthly['date_block_num']
y = monthly['item_cnt_day']

plt.ylabel('Monthly Sales', fontsize = 20)
plt.xlabel('Date Codes', fontsize = 20)

ax.bar(x, y)
plt.show()

In [ ]:
#Getting total sales of a shop for a unique date
Rossmann_sales_train_df.groupby(['shop_id','date_block_num']).sum('item_cnt_day')

In [ ]:
#Getting total sales of a shop for a unique item and date
#Transforming sales training set to required format as test.csv
Rossmann_sales_train_df_groupby=Rossmann_sales_train_df.groupby(['shop_id','item_id','date_block_num'])['item_cnt_day'].sum().reset_index()

In [ ]:
Rossmann_sales_train_df_groupby

# **Forest Cover Type**

In [ ]:
#distribution of Cover Types
plt.figure(figsize=(6,5))
plt.hist(Cover_Type_train_df['Cover_Type'])
plt.show

In [ ]:
Cover_Type_train_df.groupby(['Cover_Type']).nunique()

In [ ]:
Cover_Type_train_df.corr()

In [ ]:
cor = Cover_Type_train_df.corr()

plt.figure(figsize = (40,40))
sns.heatmap(cor, annot = True)

In [ ]:
Cover_Type_train_df.groupby(['Horizontal_Distance_To_Hydrology']).nunique()

In [ ]:
#Distribution of Horizontal Distance To Hydrology
plt.figure(figsize=(6,5))
plt.hist(Cover_Type_train_df['Horizontal_Distance_To_Hydrology'])
plt.show

In [ ]:
Cover_Type_train_df.groupby(['Hillshade_Noon']).nunique()

In [ ]:
#Distribution of Hillside Noon
plt.figure(figsize=(6,5))
plt.hist(Cover_Type_train_df['Hillshade_Noon'])
plt.show

In [ ]:
Cover_Type_train_df.groupby(['Horizontal_Distance_To_Hydrology','Hillshade_Noon']).nunique()

# **Eye Movements**

In [ ]:
#distribution of Labels
plt.figure(figsize=(6,5))
plt.hist(Eye_Movements_train_df['label'])
plt.show

In [ ]:
Eye_Movements_train_df.groupby(['label']).nunique()

In [ ]:
Eye_Movements_train_df.corr()

In [ ]:
cor = Eye_Movements_train_df.corr()

plt.figure(figsize = (30,30))
sns.heatmap(cor, annot = True)

# **Higgs Boson**

In [ ]:
#distribution of Label
plt.figure(figsize=(6,5))
plt.hist(Higgs_Boson_train_df['Label'])
plt.show

In [ ]:
Higgs_Boson_train_df.groupby(['Label']).nunique()

In [ ]:
Higgs_Boson_train_df.corr()

In [ ]:
%matplotlib inline

cor = Higgs_Boson_train_df.corr()

plt.figure(figsize = (30,30))
sns.heatmap(cor, annot = True)

# **Churn Modelling**

In [ ]:
#distribution of Exited
plt.figure(figsize=(6,5))
plt.hist(Churn_Modelling_train_df['Exited'])
plt.show

In [ ]:
Churn_Modelling_train_df.groupby(['Exited']).nunique()

In [ ]:
Churn_Modelling_train_df.corr()

In [ ]:
cor = Churn_Modelling_train_df.corr()

plt.figure(figsize = (10,10))
sns.heatmap(cor, annot = True)

# **Outliers**

# **Rossmann**

In [ ]:
plt.figure(figsize=(10,4))
plt.xlim(-100,3000)
sns.boxplot(x=Rossmann_sales_train_df.item_cnt_day)

plt.figure(figsize=(10,4))
plt.xlim(Rossmann_sales_train_df.item_price.min(), Rossmann_sales_train_df.item_price.max()*1.1)
sns.boxplot(x=Rossmann_sales_train_df.item_price)

# **XGBoost Model**

# **Rossmann**

In [ ]:
Rossmann_item_cat_joined_df=Rossmann_items_df.merge(Rossmann_item_categories_df, on='item_category_id')
Rossmann_sales_train_df_groupby_cat_join=Rossmann_sales_train_df_groupby.merge(Rossmann_item_cat_joined_df, on='item_id')
Rossmann_sales_train_df_groupby_cat_join

In [ ]:
Xgboost_X_train=Rossmann_sales_train_df_groupby_cat_join[['date_block_num','shop_id','item_id','item_category_id']]
Xgboost_X_train

In [ ]:
Xgboost_Y_train=Rossmann_sales_train_df_groupby_cat_join[['item_cnt_day']]
Xgboost_Y_train

In [ ]:
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(Xgboost_X_train, Xgboost_Y_train, test_size=0.2, random_state=123)

In [ ]:
xg_reg = xgb.XGBRegressor(objective ='reg:squarederror',max_depth = 5,  n_estimators = 100)
xg_reg.fit(X_train_xgb,y_train_xgb)
# example on how to use feature importance
xgb.plot_importance(xg_reg)
plt.figure(figsize = (16, 12))
plt.show()

In [ ]:
# visualize the model
fig, ax = plt.subplots(figsize=(60, 60))
xgb.plot_tree(xg_reg, ax=ax)
plt.savefig("sales.pdf")

In [ ]:
preds = xg_reg.predict(X_test_xgb)
rmse = np.sqrt(mean_squared_error(y_test_xgb, preds))
print("RMSE: %f" % (rmse))

# **Forest Cover Type**

In [ ]:
# Xgboost_X_train=Cover_Type_train_df[['Horizontal_Distance_To_Hydrology','Hillshade_Noon']]
Xgboost_X_train = Cover_Type_train_df.drop(columns=['Cover_Type'])
Xgboost_X_train

In [ ]:
Xgboost_Y_train=Cover_Type_train_df[['Cover_Type']]
Xgboost_Y_train['Cover_Type']=Xgboost_Y_train['Cover_Type']-1
Xgboost_Y_train

In [ ]:
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(Xgboost_X_train, Xgboost_Y_train, test_size=0.2, random_state=123)

In [ ]:
xg_reg = xgb.XGBClassifier()
xg_reg.fit(X_train_xgb,y_train_xgb)
# example on how to use feature importance
ax1 = xgb.plot_importance(xg_reg,max_num_features=10)
ax1.figure.tight_layout()
ax1.figure.savefig('forest_cover_feature_importance.png')


In [ ]:
# visualize the model
fig, ax = plt.subplots(figsize=(60, 60))
xgb.plot_tree(xg_reg, ax=ax)
plt.savefig("forest.pdf")

In [ ]:
preds = xg_reg.predict(X_test_xgb)
predictions = [round(value) for value in preds]

# evaluate predictions
accuracy = accuracy_score(y_test_xgb, predictions)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

# **Eye Movements**

In [ ]:
Xgboost_X_train=Eye_Movements_train_df.drop(columns=['label'])
Xgboost_X_train

In [ ]:
Xgboost_Y_train=Eye_Movements_train_df[['label']]
Xgboost_Y_train

In [ ]:
rmses = []
# 80 20 train test split , 
rng_seeds = [123] # implementing 5-fold cross validation with 80-20 splits. Can test with many seeds: [123, 234, 456, 678, 890]

for seed in rng_seeds:
    X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(Xgboost_X_train, Xgboost_Y_train, test_size=0.2, random_state=seed) # random_state = seed
    xg_reg = xgb.XGBClassifier()
    xg_reg.fit(X_train_xgb,y_train_xgb)
    preds = xg_reg.predict(X_test_xgb)
    predictions = [round(value) for value in preds]

    # evaluate predictions
    accuracy = accuracy_score(y_test_xgb, predictions)
    print("Accuracy: %.2f%%" % (accuracy * 100.0))

# outside loop
# final_score = np.mean(rmses)
# print("Final Score: %f" % final_score)

# example on how to use feature importance
xgb.plot_importance(xg_reg)
plt.figure(figsize = (16, 12))
plt.show()

In [ ]:
# visualize the model
fig, ax = plt.subplots(figsize=(60, 60))
xgb.plot_tree(xg_reg, ax=ax)
plt.savefig("eye.pdf")

In [ ]:
for seed in rng_seeds:

    # create a DMatrix for cross validation
    xgb_data = xgb.DMatrix(Xgboost_X_train.values, Xgboost_Y_train.values)

    params = {"objective":"reg:squarederror",'colsample_bytree': 0.3,'learning_rate': 0.1,
                'max_depth': 5, 'alpha': 10}

    xgb_cv = cv(dtrain=xgb_data, params=params, nfold=5,
                    num_boost_round=50, early_stopping_rounds=10, metrics="rmse", as_pandas=True, seed=seed)

print(xgb_cv)

# **Higgs Boson**

In [ ]:
# Xgboost_X_train=Higgs_Boson_train_df[['PRI_tau_pt','PRI_tau_phi']]
Xgboost_X_train=Higgs_Boson_train_df.drop(columns=['Label','Weight'])
Xgboost_X_train

In [ ]:
Xgboost_Y_train=Higgs_Boson_train_df[['Label']]
# Xgboost_Y_train ###### edit labels to integers
Xgboost_Y_train = Xgboost_Y_train.replace(["b","s"],[0,1])
Xgboost_Y_train

In [ ]:
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(Xgboost_X_train, Xgboost_Y_train, test_size=0.2, random_state=123)

In [ ]:
xg_reg = xgb.XGBClassifier()
xg_reg.fit(X_train_xgb,y_train_xgb)

# example on how to use feature importance
xgb.plot_importance(xg_reg)
plt.figure(figsize = (16, 12))
plt.show()

In [ ]:
# visualize the model
fig, ax = plt.subplots(figsize=(60, 60))
xgb.plot_tree(xg_reg, ax=ax)
plt.savefig("higgs.pdf")

In [ ]:
# score classification model
preds = xg_reg.predict(X_test_xgb)
predictions = [round(value) for value in preds]

# evaluate predictions
accuracy = accuracy_score(y_test_xgb, predictions)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

# **Churn Modelling**

In [ ]:
Xgboost_X_train=Churn_Modelling_train_df.drop(columns=['Exited'])


# need to convert categoricals for xgboost Surname, Geography, Gender
lbl = preprocessing.LabelEncoder()
Xgboost_X_train['Surname'] = lbl.fit_transform(Xgboost_X_train['Surname'].astype(str))
Xgboost_X_train['Geography'] = lbl.fit_transform(Xgboost_X_train['Geography'].astype(str))
Xgboost_X_train['Gender'] = lbl.fit_transform(Xgboost_X_train['Gender'].astype(str))

Xgboost_X_train


In [ ]:
Xgboost_Y_train=Churn_Modelling_train_df[['Exited']]
Xgboost_Y_train

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 80 20 train test split

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(Xgboost_X_train, Xgboost_Y_train, test_size=0.2, random_state=123)

In [ ]:
xg_reg = xgb.XGBClassifier()
xg_reg.fit(X_train_xgb,y_train_xgb)

# example on how to use feature importance
xgb.plot_importance(xg_reg)
plt.figure(figsize = (16, 12))
plt.show()

In [ ]:
# visualize the model
fig, ax = plt.subplots(figsize=(60, 60))
xgb.plot_tree(xg_reg, ax=ax)
plt.savefig("churn.pdf")

In [ ]:
preds = xg_reg.predict(X_test_xgb)
predictions = [round(value) for value in preds]

# evaluate predictions
accuracy = accuracy_score(y_test_xgb, predictions)
print("Accuracy: %.2f%%" % (accuracy * 100.0))